# تحلیل تعاملی نتایج استدلال ساختارمند

این Notebook برای تحلیل اجراهای ذخیره‌شده در GitHub Codespaces است و مدل را بارگذاری یا اجرا نمی‌کند.

می‌توانید هر اجرای موجود در پوشه‌های results و runs را انتخاب کنید و صورت سؤال، پاسخ مرجع، خروجی سه روش، مسیرهای کاندید و معیارهای کارایی را بررسی کنید.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists(): ROOT = ROOT.parent
folders = []
for base in [ROOT / 'results', ROOT / 'runs']:
    if base.exists():
        candidates = [base] if base.name == 'results' else [p for p in base.iterdir() if p.is_dir()]
        for folder in candidates:
            if (folder / 'summary.csv').exists() and (folder / 'experiment_results.jsonl').exists(): folders.append(folder)
run_options = [('پایلوت اولیه' if p.name == 'results' else p.name, str(p)) for p in folders]
if not run_options: raise FileNotFoundError('هیچ اجرای کاملی در results یا runs پیدا نشد.')
run_picker = widgets.Dropdown(options=run_options, description='انتخاب اجرا:', layout=widgets.Layout(width='80%'))
display(run_picker)

In [ ]:
def load_selected_run(_=None):
    global RUN_DIR, summary, details, config
    RUN_DIR = Path(run_picker.value)
    summary = pd.read_csv(RUN_DIR / 'summary.csv')
    records = [json.loads(line) for line in (RUN_DIR / 'experiment_results.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
    details = pd.DataFrame(records)
    cfg = RUN_DIR / 'run_config.json'
    config = json.loads(cfg.read_text(encoding='utf-8')) if cfg.exists() else {}
    display(Markdown(f'## اجرای انتخاب‌شده: {RUN_DIR.name}'))
    display(pd.Series(config, name='value').to_frame())
    display(summary)
run_button = widgets.Button(description='بارگذاری نتایج این اجرا', button_style='primary')
run_output = widgets.Output()
def on_load(_):
    with run_output:
        run_output.clear_output()
        load_selected_run()
run_button.on_click(on_load)
display(run_button, run_output)

In [ ]:
if 'summary' not in globals(): load_selected_run()
display(Markdown('## نمودارهای مقایسه'))
for field, title, multiplier in [('accuracy','دقت پاسخ نهایی (%)',100),('avg_input_tokens','میانگین توکن ورودی',1),('avg_generated_tokens','میانگین توکن خروجی',1),('avg_generation_calls','میانگین فراخوانی مدل',1),('avg_elapsed_seconds','زمان متوسط تولید (ثانیه)',1)]:
    if field in summary.columns:
        plot_data = summary.copy(); plot_data['value'] = plot_data[field] * multiplier
        fig = px.bar(plot_data, x='method', y='value', color='method', text_auto='.2f', title=title)
        if field == 'accuracy': fig.update_yaxes(range=[0,100])
        fig.show()

In [ ]:
display(Markdown('## بررسی کامل سؤال‌ها'))
question_ids = sorted(details['dataset_index'].astype(int).unique().tolist())
question_picker = widgets.Dropdown(options=question_ids, description='شناسه سؤال:')
method_picker = widgets.Dropdown(options=['Direct','CoT','ToT-inspired'], value='CoT', description='روش:')
question_output = widgets.Output()
def show_question(_=None):
    with question_output:
        question_output.clear_output()
        row = details[(details.dataset_index.astype(int) == int(question_picker.value)) & (details.method.astype(str) == method_picker.value)].iloc[0]
        display(Markdown('### صورت کامل مسئله'))
        display(Markdown(row.get('question','')))
        print('پاسخ مرجع:', row.get('gold_answer'))
        print('پاسخ مدل:', row.get('predicted_answer'))
        print('درست:', bool(row.get('correct')))
        print('توکن ورودی:', row.get('input_tokens'), '| توکن خروجی:', row.get('generated_tokens'))
        print('فراخوانی:', row.get('generation_calls'), '| زمان:', row.get('elapsed_seconds'))
        display(Markdown('#### پرامپت'))
        print(row.get('prompt',''))
        if method_picker.value == 'ToT-inspired':
            try: raw = json.loads(row.get('raw_output','{}'))
            except json.JSONDecodeError: raw = {}
            for i, candidate in enumerate(raw.get('candidates',[]),1):
                display(Markdown(f'#### Candidate {i}')); print(candidate)
            display(Markdown('#### Judge output')); print(raw.get('judge_output',''))
            display(Markdown('#### Selected candidate')); print(raw.get('selected_candidate_output',''))
        else:
            display(Markdown('#### خروجی خام مدل')); print(row.get('raw_output',''))
question_picker.observe(show_question, names='value')
method_picker.observe(show_question, names='value')
display(widgets.HBox([question_picker,method_picker]), question_output)
show_question()

In [ ]:
display(Markdown('## تحلیل اختصاصی ToT-inspired'))
tot = details[details.method.astype(str) == 'ToT-inspired'].copy()
if not tot.empty:
    valid = int(tot['judge_selection_valid'].fillna(False).astype(bool).sum())
    print(f'قالب داوری معتبر: {valid} از {len(tot)}')
    print(f'خطای قالب داوری: {len(tot)-valid}')
    display(tot[['dataset_index','gold_answer','predicted_answer','candidate_answers','selected_candidate','judge_selection_valid','correct','input_tokens','generated_tokens','elapsed_seconds']])
else: print('رکورد ToT-inspired موجود نیست.')

## ملاحظات علمی

- هر گزارش فقط مربوط به اجرای انتخاب‌شده است، نه کل مجموعه GSM8K.
- تاریخچه سؤال‌ها در حالت پیش‌فرض از تکرار شناسه‌های قبلی جلوگیری می‌کند.
- ToT-inspired پیاده‌سازی کامل Tree of Thoughts نیست.
- تولیدکننده و داور از یک مدل استفاده می‌کنند.
- توکن‌ها شاخص حجم ورودی و خروجی‌اند و معیار مستقیم هزینه مالی یا انرژی نیستند.